# Boltz processing pipeline

#### Pipeline [SMILES, CheMBL protein ID] -> yaml

In [2]:
SAFETYSCREEN9_TARGETS = {
    # GPCRs
    "Serotonin_5HT2A":  "P28223",  # Gq-coupled, CNS, psychiatric liability
    "Dopamine_D2":       "P14416",  # Gi-coupled, CNS, EPS liability
    "Muscarinic_M2":     "P08172",  # Gi-coupled, cardiac/ANS liability
    "Adrenoceptor_a1":   "P35348",  # Gq-coupled, cardiovascular liability
    # Ion Channels
    "hERG":              "Q12809",  # Voltage-gated K+, cardinal cardiac safety target
    "NMDA_NR1":          "Q05586",  # Ligand-gated, glutamate/CNS
    # Kinase
    "LCK":               "P06239",  # Src-family kinase, immunotoxicity
    # Enzyme
    "COX1":              "P23219",  # Cyclooxygenase, GI/platelet liability
    # Nuclear Receptor
    "GR":                "P04150",  # Glucocorticoid receptor, metabolic/immune liability
}

SAFETYSCREEN44_TARGETS = {
    "Opioid_Delta": "P41143",
    "Opioid_Mu": "P35372",
    "Histamine_H2": "P25021",
    "Adrenoceptor_a2": "P08913",
    "Serotonin_5HT1A":  "P08908",
    "Dopamine_D2": "P14416",
    "Serotonin_5HT2B": "P41595",
    "Vasopressin_V1A": "P37288",
    "Potassium_KV": "Q09470",
    "Adrenoceptor_b1": "P08588",
    "Adrenoceptor_b2": "P07550",
    "Adrenoceptor_a1": "P35348",
    "nAChR_a4b2": "P43681",
    "NET": "P23975",
    "AChE": "P22303",
    "Cannabinoid_CB2": "P34972",
    "Cholecystokinin_CCK1": "P32238",
    "Adenosine_A2A": "P29274",
    "PDE3A": "Q14432",
    "PDE4D2": "Q08499",
    "hERG": "Q12809",
    "Serotonin_5HT3": "P46098",
    "Serotonin_5HT1B": "P28222",
    "SERT": "P31645",
    "Dopamine_D1": "P21728",
    "MAO_A": "P21397",
    "KOP": "P41145",
    "GABAA_a1": "P14867",
    "GABAA_b2": "P47870",
    "GABAA_g2": "P18507",
    "GR": "P04150",
    "Cannabinoid_CB1": "P21554",
    "Serotonin_5HT2A": "P28223",
    "Dopamine_D1": "Q01959",
    "Endothelin_ETA": "P25101",
    # "CACNA_1C": "Q13936",
    "SCN5A": "Q14524",
    "LCK": "P06239",
    "NMDA_NR1": "Q05586",
    "NMDA_NR2A": "Q12879",
    "Histamine_H1": "P35367",
    "Muscarinic_M1": "P11229",
    "Muscarinic_M2": "P08172",
    "Muscarinic_M3": "P20309",
    "AR": "P10275",
    "COX1": "P23219",
    "COX2": "P35354"
}

In [ ]:
import requests
import yaml

def fetch_target_metadata(target_chembl_id: str) -> dict:
    url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_chembl_id}.json"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()

    components = []
    for comp in data.get("target_components", []):
        synonyms = {syn["syn_type"]: syn["component_synonym"]
                    for syn in comp.get("target_component_synonyms", [])}
        
        # Pick the canonical UniProt accession (not AlphaFold or secondary)
        accession = None
        for xref in comp.get("target_component_xrefs", []):
            if xref["xref_src_db"] == "UniProt":
                accession = xref["xref_id"]
                break
        # Fallback to comp["accession"] if no UniProt xref found
        if accession is None:
            accession = comp.get("accession")

        components.append({
            "accession":      accession,
            "component_type": comp["component_type"],
            "gene_name":      synonyms.get("GENE_SYMBOL"),
            "description":    comp.get("component_description"),
        })

    return {
        "target_chembl_id": target_chembl_id,
        "target_type":      data["target_type"],
        "pref_name":        data["pref_name"],
        "organism":         data["organism"],
        "components":       components,
    }

def fetch_uniprot_sequence(uniprot_accession: str) -> dict:
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_accession}.json"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    sequence = data["sequence"]["value"]

    return {
        "accession":        uniprot_accession,
        "sequence":         sequence,
        "length":           data["sequence"]["length"]
    }


def build_boltz_yaml(
    protein_sequence: str,
    ligand_smiles: str,
    protein_chain: str = "A",
    ligand_chain: str = "B",
    protein_name: str = "undefined",
) -> dict:
    config = {
        "version": 1,
        "sequences": [
            {
                "protein": {
                    "id": protein_chain,
                    "sequence": protein_sequence,
                    "msa": f"/path/to/your/msa/{protein_name}.a3m" # TODO replace with actual MSA path if available
                }
            },
            {
                "ligand": {
                    "id": ligand_chain,
                    "smiles": ligand_smiles,
                }
            }
        ],
        "properties": [
            {"affinity": {"binder": ligand_chain}}
        ]
    }
    return config

In [ ]:
import pandas as pd
from pathlib import Path
import yaml

UNIPROT_CACHE = {}

def generate_yamls(
    molecule_smiles: str,
    dataset_idx: str,
    output_dir: str = "boltz_inputs"
):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    results = []

    for protein_name, accession in SAFETYSCREEN9_TARGETS.items():
        out_file = output_path / f"{protein_name}_{dataset_idx}.yaml"

        if out_file.exists():
            results.append({"protein": protein_name, "status": "skipped", "path": str(out_file)})
            continue
        
        try:
            if accession in UNIPROT_CACHE:
                uniprot_data = UNIPROT_CACHE[accession]
            else:
                uniprot_data = fetch_uniprot_sequence(accession)
                UNIPROT_CACHE[accession] = uniprot_data 

            if not uniprot_data.get("sequence"):
                results.append({"protein": protein_name, "status": "no_sequence", "path": None})
                continue

            # Build YAML config
            config = build_boltz_yaml(
                protein_sequence=uniprot_data["sequence"],
                ligand_smiles=molecule_smiles,
                protein_name=protein_name                
            )

            # Write to file
            with open(out_file, "w") as f:
                yaml.dump(config, f, default_flow_style=False, sort_keys=False)

            results.append({
                "protein": protein_name,
                "uniprot": accession,
                "seq_len": uniprot_data["length"],
                "status": "ok",
                "path": str(out_file),
            })

        except Exception as e:
            print(f"  [ERROR] {protein_name}: {e}")
            results.append({"protein": protein_name, "status": f"error: {e}", "path": None})

    return pd.DataFrame(results)

Generate yamls for Chembl subset

In [ ]:
import pandas as pd 

df = pd.read_csv("<path>/chembl.csv")
df.head()

,Unnamed: 0,ChEMBL ID,QED Weighted,Smiles,apol,arorings,ASA,ast_fraglike,ast_fraglike_ext,ast_violation,...,vsa_other,vsa_pol,vsurf_G,vsurf_R,Weight,weinerPath,weinerPol,zagreb,smiles_processed,conformer_path
0,0,CHEMBL178234,0.75,Clc1sc(S(=O)(=O)n2cc(C=3CCNCC=3)c3ccccc23)cc1,52.372688,3,548.69739,0,0,1,...,49.761135,37.699100,1.307385,1.414403,379.91199,1231,40,134,[H]C1=C(c2c([H])n(S(=O)(=O)c3sc(Cl)c([H])c3[H]...,/scratch/cds/MolML/JEPA/conformers/chembl_CHEM...
1,1,CHEMBL3458119,0.91,O=C(NC(C)(C)C)c1cc(CN2CCS(=O)(=O)CC2)ccc1,51.669033,1,540.91876,0,0,2,...,19.462330,51.266022,1.331438,1.362374,324.44501,1169,29,114,[H]c1c([H])c(C(=O)N([H])C(C([H])([H])[H])(C([H...,/scratch/cds/MolML/JEPA/conformers/chembl_CHEM...
2,2,CHEMBL3787405,0.63,Oc1ccc([C@H]2[C@H](c3cc(OC)cc(OC)c3)C(=O)c3cc(...,64.815033,3,676.21198,0,0,3,...,8.458519,37.148865,1.390776,1.426751,420.46100,2377,56,166,[H]Oc1c([H])c([H])c([C@]2([H])c3c(OC([H])([H])...,/scratch/cds/MolML/JEPA/conformers/chembl_CHEM...
3,3,CHEMBL3309754,0.13,Brc1c(N2C(=O)c3ccccc3C(C(=O)Nc3cc(F)c(Oc4ccnc5...,108.855130,5,1018.69800,0,0,3,...,62.101425,52.927547,1.537090,1.509696,767.67596,13088,91,280,[H]c1nc2c([H])c(OC([H])([H])C([H])([H])C([H])(...,/scratch/cds/MolML/JEPA/conformers/chembl_CHEM...
4,4,CHEMBL1356585,0.74,Clc1cc(C(O)=O)c(Nc2oc3ccc(Cl)cc3n2)cc1,38.273552,3,495.54315,0,0,2,...,49.810867,38.498993,1.304640,1.358567,322.12698,951,31,112,[H]c1c([H])c(N([H])c2nc3c([H])c(Cl)c([H])c([H]...,/scratch/cds/MolML/JEPA/conformers/chembl_CHEM...


In [ ]:
from tqdm import tqdm

subset = df.sample(100000)

output_base_dir = "<path>/yamls"

for i in tqdm(range(1000)):
    output_dir = f"{output_base_dir}/yamls_{i}"
    subset_chunk = subset.iloc[i * 100:(i + 1) * 100] 

    for _, row in subset_chunk.iterrows():
        generate_yamls(
            molecule_smiles=row["Smiles"],
            dataset_idx=row["ChEMBL ID"],
            output_dir=output_dir,
        )

subset.to_csv("boltz_chembl_100k.csv", index=False)

100%|██████████| 1000/1000 [1:12:53<00:00,  4.37s/it]


### Adding prediction data

In [5]:
import json
import glob
import os
from tqdm import tqdm

# Directory containing all yamls
yamls_base_dir = "/home/rottach/phd/p3_jepa/boltz/boltz_results"

# Collect all JSON files
all_json_files = []
for yaml_idx in tqdm(range(2001)):  # 0 to 2000
    pattern = f"{yamls_base_dir}/yamls_{yaml_idx}/**/affinity_*.json"
    all_json_files.extend(glob.glob(pattern, recursive=True))

print(f"Found {len(all_json_files)} JSON files")

def extract_chembl_id(json_path):
    """Extract ChEMBL ID from filename like affinity_ProteinName_CHEMBL123456.json"""
    basename = os.path.basename(json_path).replace(".json", "")
    parts = basename.split("_")
    for part in reversed(parts):
        if part.startswith("CHEMBL"):
            return part
    return None

# Extract predictions
predictions = []
for json_file in tqdm(all_json_files):
    try:
        with open(json_file, "r") as f:
            data = json.load(f)
        
        predictions.append({
            "json_path": json_file,
            "chembl_id": extract_chembl_id(json_file),
            "affinity_pred_value": data.get("affinity_pred_value"),
            "affinity_probability_binary": data.get("affinity_probability_binary"),
            "affinity_pred_value1": data.get("affinity_pred_value1"),
            "affinity_probability_binary1": data.get("affinity_probability_binary1"),
        })
    except Exception as e:
        print(f"Error reading {json_file}: {e}")

df_predictions = pd.DataFrame(predictions)
df_predictions

100%|██████████| 2001/2001 [12:29<00:00,  2.67it/s]


Found 937447 JSON files


100%|██████████| 937447/937447 [22:32<00:00, 693.05it/s]  


,json_path,chembl_id,affinity_pred_value,affinity_probability_binary,affinity_pred_value1,affinity_probability_binary1
0,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL155736,0.593805,0.190625,-0.011344,0.202888
1,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL155736,0.729628,0.036124,0.914582,0.036710
2,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL155736,1.062970,0.152495,0.637991,0.200942
3,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL155736,1.136117,0.284994,0.649528,0.324720
4,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL155736,0.782734,0.106685,0.390245,0.156396
...,...,...,...,...,...,...
937442,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL458045,-0.277791,0.508228,0.636119,0.431697
937443,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL1572311,0.065481,0.410220,-0.173436,0.459479
937444,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL373132,0.196117,0.321411,0.348347,0.214883
937445,/home/rottach/phd/p3_jepa/boltz/boltz_results/...,CHEMBL3458746,2.080553,0.042156,1.497217,0.019304
